# 05 感度分析

パラメータを 1 つ（または 2 つ）ずつ変えて、**最適配分がどれだけ変わるか**を調べます（RQ3・RQ5）。
最適配分はすべて全探索（刻み 10%）で求めます。

> **補遺トピック G1・教育用の架空モデル**です。国名・経済指標・投資効果の係数はすべて架空で、現実の国家の GDP・経済成長率・政策効果を予測するものではありません。実際の政策判断に使わないでください。

## 準備：実験環境（最初に必ず実行）

G1 リポジトリの `.py`（計算の本体）を import できるように、作業ディレクトリ（G1 のルート）を自動で探して設定します。

| 実行場所 | 準備 |
| --- | --- |
| ローカルの Jupyter / VSCode | `notebooks/` の中で開けば、そのまま動きます |
| Colab（方法A：GitHub から clone） | 下のセルの `REPO_URL` に**自分たちのリポジトリの URL** を入れる |
| Colab（方法B：ZIP をアップロード） | G1 フォルダを ZIP にして `/content` にアップロードしておく（またはセルの実行中にアップロード） |

Colab の CPU ランタイムで動きます（GPU・TPU は不要）。`textbook/`（教科書のコード）は公開リポジトリに含めない約束です。`textbook/` がないと import が失敗します。

In [ ]:
#@title 準備（G1 の .py を読み込む）
import os, sys, glob, zipfile, subprocess
from pathlib import Path

# ---- 方法A：GitHub から clone する場合だけ設定する（URL は自分たちのものを入れる）----
REPO_URL = ""                                     # 例："https://github.com/<team>/<repo>.git"（空なら clone しない）
REPO_SUBDIR = "g1-economic-growth-optimization"   # リポジトリの中の G1 フォルダ（ルートが G1 なら ""）

IN_COLAB = "google.colab" in sys.modules


def is_g1_root(p):
    return (Path(p) / "economic_model.py").exists() and (Path(p) / "optimization.py").exists()


def find_g1_root():
    for p in [Path.cwd(), Path.cwd().parent]:          # ローカル：notebooks/ の中、または G1 のルート
        if is_g1_root(p):
            return p.resolve()
    if not IN_COLAB:
        return None
    if REPO_URL:                                        # Colab 方法A
        dest = Path("/content/repo")
        if not dest.exists():
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(dest)], check=True)
        cand = dest / REPO_SUBDIR if REPO_SUBDIR else dest
        if is_g1_root(cand):
            return cand
    zips = glob.glob("/content/*.zip")                  # Colab 方法B
    if not zips:
        from google.colab import files
        print("G1 フォルダの ZIP を選んでアップロードしてください")
        zips = ["/content/" + name for name in files.upload()]
    for z in zips:
        with zipfile.ZipFile(z) as zf:
            zf.extractall("/content/g1_upload")
    for hit in glob.glob("/content/g1_upload/**/economic_model.py", recursive=True):
        if is_g1_root(Path(hit).parent):
            return Path(hit).parent
    return None


G1_ROOT = find_g1_root()
if G1_ROOT is None:
    raise RuntimeError("G1 のフォルダが見つかりません。上の説明（方法A／方法B）を確認してください。")
os.chdir(G1_ROOT)
if str(G1_ROOT) not in sys.path:
    sys.path.insert(0, str(G1_ROOT))
if not (G1_ROOT / "textbook" / "fixed_queue.py").exists():
    print("⚠️ textbook/ がありません。economic_model.py・optimization.py の import が失敗します。")

# ---- 既存の .py（計算の本体）と、グラフ ----
import matplotlib.pyplot as plt
import economic_model as em
import simulation as sim
import allocation_strategies as al
import optimization as op
import evaluation as ev
import growth_charts as gc

gc.setup_style()                        # 文字を大きくするなら gc.setup_style(1.3)
PARAMS = em.load_economy()              # 架空の国家のパラメータ
SCENARIOS = al.load_scenarios()         # 比較用の配分シナリオ
LABELS = {sc["id"]: sc["label"] for sc in SCENARIOS}
YEARS = PARAMS["simulation_years"]
STEP = 0.1                              # 最適化の配分の刻み（10%）
M = al.units_count(STEP)

SAVE_FIGURES = False                    # True にすると、グラフを notebooks/figures/ に PNG で保存する
FIG_DIR = "notebooks/figures"


def show(fig, name):
    """グラフを表示し、SAVE_FIGURES = True なら PNG も保存する"""
    if SAVE_FIGURES:
        print("saved:", gc.save(fig, f"{FIG_DIR}/{name}.png"))
    plt.show()


def table(rows, columns):
    """辞書のリストを表で表示する。columns = [(キー, 見出し, 書式)]。pandas があれば DataFrame で表示"""
    try:
        import pandas as pd
        from IPython.display import display
        display(pd.DataFrame([{h: ("-" if r.get(k) is None else f.format(r[k])) for k, h, f in columns} for r in rows]))
    except ImportError:
        print(ev.format_table(rows, columns))


def alloc_cols():
    return [(s, em.SECTOR_LABELS[s], "{:.0%}") for s in em.SECTORS]


print("G1_ROOT:", G1_ROOT)
print(PARAMS["country"], "｜", YEARS, "年、年間予算", PARAMS["annual_investment_budget"], "（架空の単位）")
print("シナリオ:", [sc["label"] for sc in SCENARIOS])

## 1. 投資効率と最適配分

In [ ]:
#@title 投資効率と最適配分
OBJECTIVE = "final_gdp"          # TODO：目的関数を変える
factors = [0.5, 0.75, 1.0, 1.25, 1.5, 2.0]
shares = [op.exhaustive(op.Evaluator(em.with_changes(PARAMS, rd_efficiency=PARAMS["rd_efficiency"] * f), OBJECTIVE), M)["shares"]
          for f in factors]
show(gc.plot_share_lines(factors, shares, "R&D efficiency (x baseline)",
                         f"Optimal allocation vs R&D efficiency ({op.OBJECTIVE_LABELS[OBJECTIVE]}, model)"), "05_efficiency")

## 2. 予算規模と GDP

In [ ]:
#@title 予算規模と GDP
budgets = [25, 50, 100, 150, 200, 300]
series = {"Equal Allocation": [], "Optimized (exhaustive)": []}
for b in budgets:
    p = em.with_changes(PARAMS, annual_investment_budget=float(b))
    series["Equal Allocation"].append(sim.simulate(al.equal_allocation(), p)[-1]["gdp"])
    series["Optimized (exhaustive)"].append(op.exhaustive(op.Evaluator(p, "final_gdp"), M)["objective"])
show(gc.plot_xy(budgets, series, "annual investment budget (fictional units)", f"GDP in year {YEARS}",
                "Budget size and GDP (model)", colors={"Equal Allocation": "#3d3c39", "Optimized (exhaustive)": "#6250d6"}),
     "05_budget")

## 3. 評価期間と最適配分（RQ3）

In [ ]:
#@title 評価期間と最適配分
items = []
for objective in ["final_gdp", "final_living_standard"]:
    for years in sim.HORIZONS:
        r = op.exhaustive(op.Evaluator(PARAMS, objective, years), M)
        items.append((f"{'GDP' if objective == 'final_gdp' else 'LS'}\n{years}y", r["shares"]))
show(gc.plot_allocation_bars(items, "Optimal allocation by evaluation horizon (GDP vs living standard)"), "05_horizon")

## 4. パラメータ感度のヒートマップ

教育の効率と研究開発の効率を同時に変えたとき、GDP を最大にする配分の**研究開発の割合**がどう変わるかを示します。

In [ ]:
#@title パラメータ感度のヒートマップ
edu_f = [0.5, 1.0, 1.5, 2.0]
rd_f = [0.5, 1.0, 1.5, 2.0]
grid = []
for ef in edu_f:
    row = []
    for rf in rd_f:
        p = em.with_changes(PARAMS, education_efficiency=PARAMS["education_efficiency"] * ef,
                            rd_efficiency=PARAMS["rd_efficiency"] * rf)
        row.append(op.exhaustive(op.Evaluator(p, "final_gdp"), M)["shares"]["rd"])
    grid.append(row)
show(gc.plot_heatmap(rd_f, edu_f, grid, "R&D efficiency (x baseline)", "Education efficiency (x baseline)",
                     "Optimal R&D share for final GDP (model)", "optimal R&D share"), "05_heatmap")

## 5. 減耗率・遅延の感度

In [ ]:
rows = []
for name, ch in [("baseline", {}), ("infra depreciation 0.10", {"infrastructure_depreciation": 0.10}),
                 ("tech depreciation 0.05", {"technology_depreciation": 0.05}),
                 ("no delay", {"education_delay": 0, "rd_delay": 0}),
                 ("delay 6 years", {"education_delay": 6, "rd_delay": 6}),
                 ("returns exponent 0.8", {"returns_exponent": 0.8})]:
    p = em.with_changes(PARAMS, **ch)
    for objective in ["final_gdp", "final_living_standard"]:
        r = op.exhaustive(op.Evaluator(p, objective), M)
        rows.append({"name": f"{name} / {'GDP' if objective == 'final_gdp' else 'LS'}", "best": r["objective"], **r["shares"]})
table(rows, [("name", "change / objective", "{}"), ("best", "best value", "{:.3f}")] + alloc_cols())

## 6. 学生向け TODO

In [ ]:
# TODO 1：パラメータを変更する（下の CHANGES を書き換える）
CHANGES = {"healthcare_efficiency": PARAMS["healthcare_efficiency"] * 1.5}
# TODO 2：最適配分の変化を調べる
base = op.exhaustive(op.Evaluator(PARAMS, "weighted"), M)["shares"]
new = op.exhaustive(op.Evaluator(em.with_changes(PARAMS, **CHANGES), "weighted"), M)["shares"]
print("元     :", al.format_shares(base))
print("変更後 :", al.format_shares(new))
# TODO 3：モデルの頑健性を評価する（小さな変化で最適配分が大きく変わるパラメータはどれか）

## 考察（チームで記入）

- **TODO**：最適配分を最も大きく変えたパラメータはどれか（RQ5）
- **TODO**：短期（5 年）と長期（30 年）で最適配分が違ったか。遅延・減耗のどちらが効いているか（RQ3）
- **TODO**：パラメータの値が少し違うだけで結論が変わるなら、その結論はどこまで信頼できるか

## 参考文献

- 柴田望洋『新・明解 Pythonで学ぶアルゴリズムとデータ構造（第2版）』SBクリエイティブ（第2章 最大値、第3章 線形探索、第4章 キュー、第6章 単純挿入ソート）
- G1 リポジトリの `README.md`（モデルの仮定・計算量・モデルの限界）
- データ：`data/baseline_economy.json`、`data/investment_scenarios.json`（この教材のための**架空のデータ**）